# CropScan V2 Real-Field Training Notebook

Goal: keep the existing 38-class CropScan backend compatible, but train and evaluate with real-field images so the model is less dependent on PlantVillage lab-style backgrounds.

Datasets to download before running:

- PlantVillage: Kaggle dataset `mohitsingh1804/plantvillage` or another folder-formatted PlantVillage mirror with 38 class folders.
- PlantDoc classification: GitHub `https://github.com/pratikkayal/PlantDoc-Dataset`, or Kaggle `nirmalsankalana/plantdoc-dataset`.

Expected folder layout after download:

```text
data/
  plantvillage/PlantVillage/<PlantVillage class folders>/*.jpg
  plantdoc/train/<PlantDoc class folders>/*.jpg
  plantdoc/test/<PlantDoc class folders>/*.jpg
```

If your folder names differ, update `CONFIG` and `PLANTDOC_TO_CROPSCAN` below.

Caution: PlantDoc contains real-field imagery, but its public split can include near-duplicate scenes. Treat PlantDoc test metrics as a useful stress test, not as a final publication-grade generalization claim. For a paper, build a scene-disjoint split.

In [ ]:
from __future__ import annotations

import json
import math
import random
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

SEED = 434
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
@dataclass
class TrainConfig:
    plantvillage_dir: Path = Path("data/plantvillage/PlantVillage")
    plantdoc_train_dir: Path = Path("data/plantdoc/train")
    plantdoc_test_dir: Path = Path("data/plantdoc/test")
    artifact_dir: Path = Path("artifacts/v2")
    image_size: int = 224
    batch_size: int = 32
    epochs: int = 12
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    plantdoc_train_weight: float = 3.0
    target_confident_precision: float = 0.90

CONFIG = TrainConfig()
CONFIG.artifact_dir.mkdir(parents=True, exist_ok=True)
asdict(CONFIG)

In [ ]:
CLASS_NAMES = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper__bell___Bacterial_spot",
    "Pepper__bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch",
    "Strawberry___healthy",
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus",
    "Tomato_healthy",
]

CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

PLANTDOC_TO_CROPSCAN = {
    "Apple Scab Leaf": "Apple___Apple_scab",
    "Apple rust leaf": "Apple___Cedar_apple_rust",
    "Bell_pepper leaf spot": "Pepper__bell___Bacterial_spot",
    "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn rust leaf": "Corn_(maize)___Common_rust_",
    "Potato leaf early blight": "Potato___Early_blight",
    "Potato leaf late blight": "Potato___Late_blight",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
    "Tomato Early blight leaf": "Tomato_Early_blight",
    "Tomato Septoria leaf spot": "Tomato_Septoria_leaf_spot",
    "Tomato leaf bacterial spot": "Tomato_Bacterial_spot",
    "Tomato leaf late blight": "Tomato_Late_blight",
    "Tomato leaf mosaic virus": "Tomato__Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato mold leaf": "Tomato_Leaf_Mold",
    "Tomato two spotted spider mites leaf": "Tomato_Spider_mites_Two_spotted_spider_mite",
    "grape leaf black rot": "Grape___Black_rot",
}

# Generic PlantDoc folders such as "Tomato leaf" or "Apple leaf" are intentionally
# skipped. They do not mean lab-confirmed healthy, so mapping them to ___healthy
# would inject label noise into the healthiest-looking class.

assert set(PLANTDOC_TO_CROPSCAN.values()).issubset(CLASS_TO_IDX), "Bad PlantDoc mapping"

In [ ]:
class ImageSampleDataset(Dataset):
    def __init__(self, samples: list[tuple[Path, int]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


def collect_folder_samples(root: Path, class_map: dict[str, str]) -> list[tuple[Path, int]]:
    if not root.exists():
        print(f"Missing dataset folder: {root}")
        return []

    samples: list[tuple[Path, int]] = []
    skipped_folders = []
    for folder in sorted(path for path in root.iterdir() if path.is_dir()):
        mapped_class = class_map.get(folder.name)
        if mapped_class is None:
            skipped_folders.append(folder.name)
            continue
        label = CLASS_TO_IDX[mapped_class]
        for image_path in folder.rglob("*"):
            if image_path.suffix.lower() in IMG_EXTS:
                samples.append((image_path, label))

    if skipped_folders:
        print("Skipped unmapped folders:", skipped_folders)
    return samples


def stratified_split(samples: list[tuple[Path, int]], val_ratio=0.10, test_ratio=0.10):
    grouped = defaultdict(list)
    for sample in samples:
        grouped[sample[1]].append(sample)

    train, val, test = [], [], []
    rng = random.Random(SEED)
    for label_samples in grouped.values():
        rng.shuffle(label_samples)
        n = len(label_samples)
        n_test = max(1, int(n * test_ratio)) if n >= 10 else max(0, int(n * test_ratio))
        n_val = max(1, int(n * val_ratio)) if n >= 10 else max(0, int(n * val_ratio))
        test.extend(label_samples[:n_test])
        val.extend(label_samples[n_test:n_test + n_val])
        train.extend(label_samples[n_test + n_val:])

    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)
    return train, val, test


plantvillage_map = {name: name for name in CLASS_NAMES}
pv_samples = collect_folder_samples(CONFIG.plantvillage_dir, plantvillage_map)
pd_train_samples = collect_folder_samples(CONFIG.plantdoc_train_dir, PLANTDOC_TO_CROPSCAN)
pd_test_samples = collect_folder_samples(CONFIG.plantdoc_test_dir, PLANTDOC_TO_CROPSCAN)

pv_train, pv_val, pv_test = stratified_split(pv_samples)
pd_train_core, pd_calibration, _pd_unused = stratified_split(pd_train_samples, val_ratio=0.20, test_ratio=0.0)
train_samples = pv_train + pd_train_core
val_samples = pv_val
calibration_samples = pd_calibration if pd_calibration else pv_val
test_sets = {
    "plantvillage_test": pv_test,
    "plantdoc_test": pd_test_samples,
}

print({
    "plantvillage_total": len(pv_samples),
    "plantdoc_train_core": len(pd_train_core),
    "plantdoc_calibration": len(pd_calibration),
    "plantdoc_test": len(pd_test_samples),
    "train_total": len(train_samples),
    "val_total": len(val_samples),
    "calibration_total": len(calibration_samples),
})
assert train_samples and val_samples, "Set dataset paths before training."

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG.image_size, scale=(0.55, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.04),
    transforms.RandomAutocontrast(p=0.25),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.18), ratio=(0.3, 3.3)),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CONFIG.image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def make_weighted_sampler(samples: list[tuple[Path, int]]):
    label_counts = defaultdict(int)
    for _, label in samples:
        label_counts[label] += 1
    weights = []
    for path, label in samples:
        base_weight = 1.0 / label_counts[label]
        if "plantdoc" in str(path).lower():
            base_weight *= CONFIG.plantdoc_train_weight
        weights.append(base_weight)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(
    ImageSampleDataset(train_samples, train_transform),
    batch_size=CONFIG.batch_size,
    sampler=make_weighted_sampler(train_samples),
    num_workers=CONFIG.num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    ImageSampleDataset(val_samples, eval_transform),
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=CONFIG.num_workers,
    pin_memory=torch.cuda.is_available(),
)
calibration_loader = DataLoader(
    ImageSampleDataset(calibration_samples, eval_transform),
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=CONFIG.num_workers,
    pin_memory=torch.cuda.is_available(),
)
test_loaders = {
    name: DataLoader(
        ImageSampleDataset(samples, eval_transform),
        batch_size=CONFIG.batch_size,
        shuffle=False,
        num_workers=CONFIG.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    for name, samples in test_sets.items()
    if samples
}
list(test_loaders)

In [ ]:
def build_model(model_name: str) -> nn.Module:
    if model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(model.classifier[1].in_features, len(CLASS_NAMES)),
        )
        return model
    if model_name == "mobilenet_v2":
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V2
        model = models.mobilenet_v2(weights=weights)
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(model.classifier[1].in_features, len(CLASS_NAMES)),
        )
        return model
    raise ValueError(f"Unknown model: {model_name}")


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += labels.size(0)

    return {
        "loss": total_loss / total_count,
        "accuracy": total_correct / total_count,
    }


@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    logits_all, labels_all = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        logits_all.append(model(images).detach().cpu())
        labels_all.append(labels.cpu())
    return torch.cat(logits_all), torch.cat(labels_all)


def evaluate_logits(logits: torch.Tensor, labels: torch.Tensor, temperature: float = 1.0):
    probs = torch.softmax(logits / temperature, dim=1)
    preds = probs.argmax(dim=1)
    y_true = labels.numpy()
    y_pred = preds.numpy()
    return {
        "accuracy": float((preds == labels).float().mean().item()),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }


def train_model(model_name: str):
    model = build_model(model_name).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG.epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    best_state = None
    best_macro_f1 = -1.0
    history = []
    for epoch in range(1, CONFIG.epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        val_logits, val_labels = collect_logits(model, val_loader)
        val_metrics = evaluate_logits(val_logits, val_labels)
        scheduler.step()

        row = {"epoch": epoch, "train": train_metrics, "val": val_metrics}
        history.append(row)
        print(json.dumps(row, indent=2))

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_state = {key: value.detach().cpu() for key, value in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, history

In [ ]:
def fit_temperature(logits: torch.Tensor, labels: torch.Tensor) -> float:
    logits = logits.to(device)
    labels = labels.to(device)
    temperature = torch.ones(1, device=device, requires_grad=True)
    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=50)
    criterion = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = criterion(logits / temperature.clamp(0.5, 5.0), labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(temperature.detach().clamp(0.5, 5.0).item())


def uncertainty_features(logits: torch.Tensor, temperature: float):
    probs = torch.softmax(logits / temperature, dim=1)
    top2 = torch.topk(probs, k=2, dim=1).values
    max_prob = top2[:, 0]
    margin = top2[:, 0] - top2[:, 1]
    entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=1) / math.log(len(CLASS_NAMES))
    preds = probs.argmax(dim=1)
    return preds, max_prob, margin, entropy


def expected_calibration_error(logits: torch.Tensor, labels: torch.Tensor, temperature: float, bins: int = 15):
    preds, max_prob, _, _ = uncertainty_features(logits, temperature)
    correct = (preds == labels).float()
    ece = torch.tensor(0.0)
    for lower in torch.linspace(0, 1, bins + 1)[:-1]:
        upper = lower + 1 / bins
        mask = (max_prob > lower) & (max_prob <= upper)
        if mask.any():
            confidence = max_prob[mask].mean()
            accuracy = correct[mask].mean()
            ece += mask.float().mean() * torch.abs(confidence - accuracy)
    return float(ece.item())


def search_abstention_thresholds(logits: torch.Tensor, labels: torch.Tensor, temperature: float):
    preds, max_prob, margin, entropy = uncertainty_features(logits, temperature)
    correct = preds == labels
    best = None
    for conf_t in np.linspace(0.45, 0.95, 11):
        for margin_t in np.linspace(0.05, 0.45, 9):
            for entropy_t in np.linspace(0.25, 0.85, 13):
                mask = (max_prob >= conf_t) & (margin >= margin_t) & (entropy <= entropy_t)
                coverage = float(mask.float().mean().item())
                if mask.sum().item() < 20:
                    continue
                precision = float(correct[mask].float().mean().item())
                candidate = {
                    "max_prob_min": round(float(conf_t), 4),
                    "margin_min": round(float(margin_t), 4),
                    "entropy_max": round(float(entropy_t), 4),
                    "confident_precision": round(precision, 4),
                    "confident_coverage": round(coverage, 4),
                }
                if precision >= CONFIG.target_confident_precision:
                    if best is None or coverage > best["confident_coverage"]:
                        best = candidate

    return best or {
        "max_prob_min": 0.70,
        "margin_min": 0.20,
        "entropy_max": 0.55,
        "confident_precision": None,
        "confident_coverage": None,
    }

In [ ]:
results = {}
for model_name in ["efficientnet_b0", "mobilenet_v2"]:
    print(f"Training {model_name}")
    model, history = train_model(model_name)
    val_logits, val_labels = collect_logits(model, val_loader)
    calibration_logits, calibration_labels = collect_logits(model, calibration_loader)
    temperature = fit_temperature(val_logits, val_labels)
    thresholds = search_abstention_thresholds(calibration_logits, calibration_labels, temperature)

    model_results = {
        "history": history,
        "temperature": temperature,
        "val_metrics": evaluate_logits(val_logits, val_labels, temperature),
        "val_ece": expected_calibration_error(val_logits, val_labels, temperature),
        "calibration_metrics": evaluate_logits(calibration_logits, calibration_labels, temperature),
        "calibration_ece": expected_calibration_error(calibration_logits, calibration_labels, temperature),
        "abstention_thresholds": thresholds,
        "test_metrics": {},
    }

    for test_name, loader in test_loaders.items():
        logits, labels = collect_logits(model, loader)
        metrics = evaluate_logits(logits, labels, temperature)
        metrics["ece"] = expected_calibration_error(logits, labels, temperature)
        model_results["test_metrics"][test_name] = metrics
        print(model_name, test_name, json.dumps(metrics, indent=2))

    weight_path = CONFIG.artifact_dir / f"{model_name}_cropscan_v2.pth"
    torch.save(model.state_dict(), weight_path)
    model_results["weight_path"] = str(weight_path)
    bundle_path = CONFIG.artifact_dir / f"{model_name}_cropscan_v2_bundle.pt"
    torch.save({
        "state_dict": model.state_dict(),
        "temperature": temperature,
        "thresholds": thresholds,
        "class_names": CLASS_NAMES,
        "version": "cropscan-v2",
        "trained_on": {
            "plantvillage_train": len(pv_train),
            "plantdoc_train_core": len(pd_train_core),
            "plantdoc_calibration": len(pd_calibration),
        },
        "metrics": {key: value for key, value in model_results.items() if key.endswith("metrics") or key.endswith("ece")},
    }, bundle_path)
    model_results["bundle_path"] = str(bundle_path)
    results[model_name] = model_results

with open(CONFIG.artifact_dir / "labels.json", "w", encoding="utf-8") as file:
    json.dump(CLASS_NAMES, file, indent=2)

with open(CONFIG.artifact_dir / "training_report.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2)

results

In [ ]:
# Optional offline/mobile artifact for the smaller model.
# This is not a full offline app, but it gives the frontend/mobile track a deployable model artifact later.
mobilenet_path = CONFIG.artifact_dir / "mobilenet_v2_cropscan_v2.pth"
if mobilenet_path.exists():
    mobile_model = build_model("mobilenet_v2").to(device)
    mobile_model.load_state_dict(torch.load(mobilenet_path, map_location=device))
    mobile_model.eval()
    example = torch.randn(1, 3, CONFIG.image_size, CONFIG.image_size, device=device)
    traced = torch.jit.trace(mobile_model, example)
    traced.save(str(CONFIG.artifact_dir / "mobilenet_v2_cropscan_v2_torchscript.pt"))
    print("Saved TorchScript model for future offline testing.")

## How to use the output in the backend

After training, copy these files into `backend/models` or your deployment model artifact store:

- `efficientnet_b0_cropscan_v2.pth`
- `mobilenet_v2_cropscan_v2.pth`
- `efficientnet_b0_cropscan_v2_bundle.pt`
- `mobilenet_v2_cropscan_v2_bundle.pt`
- `labels.json`
- `training_report.json`

Backend integration after training:

1. Copy `training_report.json` into `backend/models` so inference can load temperature and abstention thresholds.
2. Copy the `.pth` files into `backend/models`, either replacing the current filenames or updating backend settings.
3. Keep PlantDoc test metrics separate from PlantVillage metrics in the report. That is the evidence that the real-field problem is being handled honestly.
4. Prefer the `_bundle.pt` artifacts for a Hugging Face release because they include model card metadata in one file.